In [ ]:
%run samples.ipynb

## __Задание 2__

####  1. Энтропийное кодирование
* Написать функцию расчета энтропии сообщения в зависимости от длины кода символа.\
 __Входные данные:__ байтовая строка и длина кода символов в байта(равномерное кодирование) \
__Выходные данные:__ значение энтропии. \


In [ ]:
from collections import Counter
import numpy as np


def calculate_entropy(data: bytes, ms: int = 1) -> float:
        """Расчет энтропии для символов в ms байт"""
        if not data:
            return 0.0

        n = len(data)
        if n % ms != 0:
            # дополняем до кратности ms
            data += b'\x00' * (ms - (n % ms))
            n = len(data)

        symbols = [data[i : i + ms] for i in range(0, n, ms)]
        counts = Counter(symbols)

        return sum(( -( p := count / len(symbols)) * np.log2(p) for count in counts.values()))

Взять любой текст на английском, отфильтровать из него текстовые символы, имеющие юникод выше 127, и исследовать зависимость значения энтропии всего сообщения от длины кода символов (1-4 байт), выведя её на график. Как можно интерпретировать эти результаты?

In [ ]:
def analyze_entropy_dependency(text: str) -> None:
    print("\nИсследование энтропии от Ms (длины кода):")
    clean_data = bytes([b for b in text.encode('ascii', 'ignore') if b < 128])
    for ms in range(1, 5):
        ent = calculate_entropy(clean_data, ms)
        print(f"Ms = {ms} байт: {ent:.4f} бит/символ (всего {ent*len(clean_data)/ms/8:.2f} байт)")

In [ ]:
DC2_PROLOGUE = "resources/small/dc2-prologue.txt"
DC_PLOT_RU = "resources/small/dc-plot-ru.txt"


with open(DC2_PROLOGUE, "r", encoding="utf-8") as f:
    analyze_entropy_dependency(f.read())

* Написать функции прямого и обратного преобразования MTF. \
__Входные и выходные данные:__ байтовая строка.

In [ ]:
class MoveToFrontTransform:
    """Преобразование Move-To-Front"""

    @staticmethod
    def encode(data: bytes) -> bytes:
        alphabet = list(range(256))
        result = bytearray()
        for byte in data:
            idx = alphabet.index(byte)
            result.append(idx)
            alphabet.insert(0, alphabet.pop(idx)) 
        return bytes(result)

    @staticmethod
    def decode(data: bytes) -> bytes:
        alphabet = list(range(256))
        result = bytearray()
        for idx in data:
            byte = alphabet.pop(idx)
            result.append(byte)
            alphabet.insert(0, byte)
        return bytes(result)


mtf = MoveToFrontTransform

In [ ]:
mtf.encode(b"abracadabra"), mtf.decode(mtf.encode(b"abracadabra"))


Как стоило бы модифицировать ваши функции, если бы вы работали с символами в кодировке UTF-8? Исследуйте влияние MTF на значение энтропии текста из тестовой выборки и сделайте вывод о применимости алгоритма перед энтропийным кодированием.

- Увеличить алфавит. Охватывать весь юникод или строить относительно текста
- Возвращать индексы а не байты, юникод больше 256 различных значений, на вход получать строку, а не байты
- Использовать l2-списки для более быстрого удаления/вставки в начало

In [ ]:
class MoveToFrontTransformUTF8:
    @staticmethod
    def encode(text: str) -> tuple[list[int], list[str]]:
        # Динамический алфавит из уникальных символов текста, 
        # отсортированный для стабильности
        alphabet = sorted(list(set(text)))
        initial_alphabet = alphabet.copy()
        result = []
        
        for char in text:
            idx = alphabet.index(char)
            result.append(idx)
            # Перемещаем символ в начало
            alphabet.insert(0, alphabet.pop(idx))
        return result, initial_alphabet

    @staticmethod
    def decode(indices: list[int], initial_alphabet: list[str]) -> str:
        alphabet = list(initial_alphabet)
        result = []
        
        for idx in indices:
            char = alphabet.pop(idx)
            result.append(char)
            alphabet.insert(0, char)
        return "".join(result)


mtf_utf8 = MoveToFrontTransformUTF8

In [ ]:
import numpy as np
from collections import Counter


def calculate_utf8_entropy(text: str) -> float:
    if not data:
        return 0.0

    n = len(text)
    if n == 0:
        return 0.0

    counts = Counter(text)
    probabilities = np.array(list(counts.values())) / n
    return -np.sum(probabilities * np.log2(probabilities))

In [ ]:
with open(DC2_PROLOGUE, "rb") as f:
    data = f.read()

print(calculate_entropy(data), calculate_entropy(mtf.encode(data)))

In [ ]:
with open(DC_PLOT_RU, "r") as f:
    data = f.read()

print(calculate_utf8_entropy(data), calculate_utf8_entropy(mtf_utf8.encode(data)[0]))

На выборке энтропися остается почти неизменной (или увеличивается). Но теперь энтропийный кодировщик будет работать с более узким набором данных, вместо всего юникода.

Перед энропийным кодировщиком однозначно полезен, если в тексте есть повторы (в языке всегда есть повторы, т.к. алфавит достаточно ограниченный, и часто встречающиеся символы будут уходить в начало быстро).

*   Написать функции, реализующую кодирование (сжатие) и декодирование с помощью алгоритма Хаффмана.\
 __Входные данные:__ функций байтовая строка и вероятностная модель источника сообщения. \
__Выходные данные:__ функций байтовая строка.\
Какие метаданные необходимо записать для правильного декодирования сжатого сообщения? Как это может повлиять на итоговый коэффициент сжатия?
Напишите дополнительные функции для записи и чтения сжатого сообщения из файла.


In [ ]:
class HuffmanCoding:
    class Node:
        def __init__(self, symbol, freq, left=None, right=None):
            self.symbol = symbol
            self.freq = freq
            self.left = left
            self.right = right

    @staticmethod
    def calculate_model(data: bytes) -> dict[bytes, int]:
        return Counter(data)

    @classmethod
    def build_tree(cls, probabilities: dict):
        # Создаем список узлов из словаря вероятностей {byte: prob}
        nodes = [cls.Node(b, p) for b, p in probabilities.items()]

        while len(nodes) > 1:
            # Сортируем вручную по частоте (замена heapq)
            nodes.sort(key=lambda x: x.freq)

            # Берем два узла с минимальной вероятностью
            left = nodes.pop(0)
            right = nodes.pop(0)

            # Создаем родительский узел
            parent = cls.Node(None, left.freq + right.freq, left, right)
            nodes.append(parent)

        return nodes[0] if nodes else None

    @classmethod
    def get_codes(cls, node, current_code="", codes=None):
        if codes is None: codes = {}
        if node is None: return codes

        # Если это лист (есть значение байта)
        if node.symbol is not None:
            codes[node.symbol] = current_code

        cls.get_codes(node.left, current_code + "0", codes)
        cls.get_codes(node.right, current_code + "1", codes)
        return codes

    @classmethod
    def encode(cls, data: bytes, probabilities: dict = None) -> bytes:
        if not data: return b""

        tree = cls.build_tree(probabilities)
        codes = cls.get_codes(tree)

        # Формируем строку бит
        bit_string = "".join(codes[b] for b in data)

        # Добавляем padding (заполнение до целого байта) и сохраняем информацию о нем
        padding = 8 - (len(bit_string) % 8)
        if padding == 8: padding = 0
        
        bit_string = bit_string + ("0" * padding)
        
        # Превращаем в байты. Первый байт будет хранить кол-во бит заполнения.
        res = bytearray()
        res.append(padding)
        for i in range(0, len(bit_string), 8):
            res.append(int(bit_string[i:i+8], 2))

        print(padding, )
            
        return bytes(res)

    @classmethod
    def decode(cls, encoded_data: bytes, probabilities: dict) -> bytes:
        if not encoded_data: return b""
        
        tree = cls.build_tree(probabilities)
        padding = encoded_data[0]
        bit_string = "".join(bin(b)[2:].zfill(8) for b in encoded_data[1:])

        # Превращаем байты в строку бит
        if padding > 0:
            bit_string = bit_string[:-padding]

        # Декодируем, проходя по дереву
        res = bytearray()
        current_node = tree
        for bit in bit_string:
            if bit == "0":
                current_node = current_node.left
            else:
                current_node = current_node.right
                
            if current_node.symbol is not None:
                res.append(current_node.symbol)
                current_node = tree
                
        return bytes(res)

    @staticmethod
    def save_compressed(filepath: str, data: bytes, probabilities: dict) -> None:
        """
        Структура файла:
        [2 байта: кол-во уникальных символов N]
        [N раз по (1 байт: символ, 4 байта: частота)]
        [Оставшееся: (1 байт паддинг) закодированные данные]
        """
        unique_chars = len(probabilities)
        
        with open(filepath, 'wb') as f:
            # H - ushort 2 байт
            f.write(struct.pack('<H', unique_chars))
            
            # B - char 1 байт I - uint 4 байт
            for char, count in probabilities.items():
                f.write(struct.pack('<BI', char, count))
            
            f.write(data)

    @staticmethod
    def load_compressed(filepath: str) -> tuple[bytes, dict]:
        with open(filepath, 'rb') as f:
            # Читаем количество уникальных символов
            header_data = f.read(2)
            if not header_data: return b""
            unique_chars = struct.unpack('<H', header_data)[0]
            
            # Восстанавливаем таблицу частот
            probabilities = {}
            for _ in range(unique_chars):
                char, count = struct.unpack('<BI', f.read(5))
                probabilities[char] = count
            
            # Все остальное — это сжатые данные
            data = f.read()
            
        return data, probabilities


huffman = HuffmanCoding

In [ ]:
import os


TEST_HUFFMAN_CYCLE = "resources/test_huffman_cycle"

def test_huffman_cycle_integrity(data: bytes = None, *, omit_prints: bool = False, keep_file: bool = False) -> bool:
    """Тестирование цикла кодирования"""
    print("--- Цикл кодирования Huffman ---")

    if data is None:
        data = b"hello huffman"

    with open(TEST_HUFFMAN_CYCLE, "wb") as f:
        f.write(data)
    written_size = os.path.getsize(TEST_HUFFMAN_CYCLE)

    probabilities = huffman.calculate_model(data)

    if not omit_prints:
        print(f"data:    {data.hex()}")
        print(f"{probabilities = }")

    huffman.save_compressed(TEST_HUFFMAN_CYCLE, huffman.encode(data, probabilities), probabilities)
    encoded_size = os.path.getsize(TEST_HUFFMAN_CYCLE)
    new_data, new_probabilities = huffman.load_compressed(TEST_HUFFMAN_CYCLE)

    if not keep_file:
        os.remove(TEST_HUFFMAN_CYCLE)

    if not omit_prints:
        print(f"compressed data: {new_data.hex()}")
        print(f"{new_probabilities = }")

    print(f"factor = {written_size / encoded_size}")

    decoded_data = huffman.decode(new_data, new_probabilities)

    integrity = data == decoded_data
    if not omit_prints:
        print(f"decoded: {decoded_data.hex()}")
        print(f"{integrity = }")

    return integrity

В качестве дополнительных метаданных будем записывать таблицу частот в "паскалевском" формате, а также количество незначащих бит в конце сообщения.
Это повлияет на на коэффициент сжатия. Чем меньше размер файла, тем заметнее влияние наличия таблицы. Для файлов больших размеров, лишний 1 Кб почти не изменит итоговую картину.

In [ ]:
test_huffman_cycle_integrity()

with open(DC2_PROLOGUE, "rb") as f:
    print(test_huffman_cycle_integrity(f.read() * 100, omit_prints=True, keep_file=True))

*   Попробовать реализовать алгоритм арифметического кодирования и опытным путем установить при каких длинах строки левая и правая граница полуинтервала начинают совпадать при хранении их в типе double.\
__Входные данные:__ байтовая строка и вероятностная модель источника сообщения.\
__Выходные данные:__ число в формате представления double.


In [ ]:
from collections import Counter

class ArithmeticCoding:
    @staticmethod
    def get_probabilities(data: bytes) -> dict:
        counts = Counter(data)
        total = len(data)
        sorted_chars = sorted(counts.keys())
        
        prob_map = {}
        low = 0.0
        for char in sorted_chars:
            prob = counts[char] / total
            prob_map[char] = (low, low + prob)
            low += prob
        return prob_map

    @staticmethod
    def encode(data: bytes, prob_map: dict) -> float:
        low = 0.0
        high = 1.0
        
        for byte in data:
            range_width = high - low
            char_low, char_high = prob_map[byte]
            
            # Обновляем границы текущего интервала
            high = low + range_width * char_high
            low = low + range_width * char_low
            
            # Проверка на совпадение границ
            if low == high:
                break
                
        # Результатом является любое число внутри итогового интервала
        return (low + high) / 2


arifm = ArithmeticCoding

In [ ]:
def find_precision_limit(data: bytes = None):
    # Тестовая строка (например, повторяющиеся байты)
    test_data = data or b"arithmetic_coding_test_data" * 10 
    # Вероятностная модель
    probs = arifm.get_probabilities(test_data)
    
    low = 0.0
    high = 1.0
    
    for i, byte in enumerate(test_data):
        range_width = high - low
        char_low, char_high = probs[byte]
        
        new_high = low + range_width * char_high
        new_low = low + range_width * char_low
        
        # Если из-за ограниченной точности double границы совпали
        if new_low == new_high:
            print(f"Границы совпали на символе номер: {i}")
            return i
            
        low, high = new_low, new_high
    
    print("Предел не достигнут на данной строке")
    return len(test_data)

import random
# Запуск эксперимента
with open("resources/experiments/arifm.txt", "w") as f:
    for i in range(1, 256):
        s = bytes((random.randint(0, i) for _ in range(2**8)))
        H = calculate_entropy(s)
        print(f"{i}: {H = } ")
        f.write(f"{H} {find_precision_limit(s)}\n")


####  2. Преобразование Барроуза-Уиллера
* Написать алгоритм прямого BWT c прямым построением матрицы Барроуза-Уиллера.\
 __Входные данные:__ байтовая строка. \
__Выходные данные:__ байтовая строка.\


In [ ]:
class BurrowsWheelerTransformFW:
    @staticmethod
    def encode(data: bytes) -> tuple[bytes, int]:
        shifts = np.array([list(data[i:] + data[:i]) for i in range(len(data))], np.uint8)
        shifts = shifts[np.lexsort(shifts.T[::-1])]

        start_idx, = np.where((shifts == np.frombuffer(data, np.uint8)).all(axis=1))[0]
        return bytes(shifts[:, -1]), start_idx

    @staticmethod
    def decode(data: bytes, start_idx: int) -> bytes:
        n = len(data)
        # решейп в столбец
        bwt_column = np.frombuffer(data, np.uint8).reshape(-1, 1)
        table = np.empty((n, 0), np.uint8)

        for _ in range(n):
            # перенос столбца вперёд
            table = np.hstack((bwt_column, table))
            table = table[np.lexsort(table.T[::-1])]

        return bytes(table[start_idx])


bwtFW = BurrowsWheelerTransformFW

In [ ]:
text = b"banana"
encoded, idx = bwtFW.encode(text)
print(f"BWT: {encoded}, idx: {idx}")
decoded = bwtFW.decode(encoded, idx)
print(f"decoded: {decoded}")

* Написать алгоритм обратного BWT с прямым построением матрицы Барроуза-Уиллера. Проверить корректность работы программы на строке “0x62 0x61 0x6e 0x61 0x6e 0x61”. \
Оценить пространственную и временную сложность написанных алгоритмов.


S(n) = O(N^2). В памяти хранится вся таблица BWT размером N x N.
T(n) = O(N^2logN). N итераций, на каждой итерации выполняется лексиграфическая сортировка, в среднем O(NlogN) в худшем сравнивает строки целиком

* Написать алгоритм обратного BWT с использованием порождаемой сдвигом и сортировкой перестановки.  Сортировку последнего столбца осуществить с помощью сортировки подсчетом.\
Оценить пространственную и временную сложность.


In [ ]:
class BurrowsWheelerTransformPermutation(
    BurrowsWheelerTransformFW
):
    @staticmethod
    def decode(data: bytes, start_idx: int) -> bytes:
        n = len(data)
        if n == 0: return b""

        counts = np.zeros(256, dtype=int)
        for symbol in data:
            counts[symbol] += 1

        first_column_pos = np.zeros(256, dtype=int)
        current_pos = 0
        for i in range(256):
            first_column_pos[i] = current_pos
            current_pos += counts[i]

        perm = np.zeros(n, dtype=int)
        temp_pos = first_column_pos.copy()
        for i in range(256):
            symbol = data[i]
            perm[i] = temp_pos[symbol]
            temp_pos[symbol] += 1

        # i-й символ первого столбца - (i - 1)-й символ исходной строки
        res = bytearray(n)
        curr = start_idx
        for i in range(n - 1, -1, -1):
            res[i] = data[curr]
            curr = perm[curr]

        return bytes(res)

$T(n) = O(N + \sum)$

- подсчёт $O(N)$
- масссив частот $O(\sum)$

$S(n) = O(N + \sum)$

- N - вектор перестановки
- $\sum$ - масссив частот


*  Последовательно применить BWT и RLE к тестовым файлам и рассчитать коэффициент сжатия. Перед этим подумать о работоспособности алгоритма прямого BWT при больших размерах файла. Дополнить функцию разбиением исходной байтовой строки на блоки. Не забыть добавить в функцию обратного преобразования блочную обработку. Удостовериться, что модификация успешно работает на enwik7 и тексте на русском языке. Проанилизировать результат.

In [ ]:
import typing as t
import struct

DEFAULT_CHUNK_SIZE = 1024 * 1024

class BurrowsWheelerTransformChunk(
    BurrowsWheelerTransformPermutation
):
    @staticmethod
    def encode(data: bytes, chunk_size: int = None) -> t.Iterator[tuple[int, bytes]]:
        chunk_size = chunk_size or DEFAULT_CHUNK_SIZE

        for i in range(0, len(data), chunk_size):
            yield super(
                BurrowsWheelerTransformChunk, BurrowsWheelerTransformChunk
            ).encode(data[i:i+chunk_size])

    @staticmethod
    def decode(encoded_chunks: t.Iterator[tuple[int, bytes]]) -> bytes:
        res = bytearray()
        for last_column, start_idx in encoded_chunks:
            res.extend(super(
                BurrowsWheelerTransformChunk, BurrowsWheelerTransformChunk
            ).decode(last_column, start_idx))

    @classmethod
    def write_decoded(cls, data: bytes, output_path: str, chunk_size: int = None):
        chunk_size = chunk_size or DEFAULT_CHUNK_SIZE
        with open(output_path, "wb") as f:
            f.write(struct.pack("I", chunk_size))

            chunk_i = 0
            for chunk, start_idx in cls.encode(data, chunk_size):
                print(chunk_i); chunk_i += 1
                f.write(struct.pack("I", start_idx))
                f.write(chunk)

    @classmethod
    def read_decoded(cls, path: str) -> bytes:
        with open(path, "rb") as f:
            chunk_size = struct.unpack("I", f.read(4))

            def generator():
                while (start_idx := struct.unpack("I", f.read(4))) != 0 and (chunk := f.read(chunk_size)) != 0:
                    yield (chunk, start_idx)

            return cls.decode(generator())
    

bwtChunk = BurrowsWheelerTransformChunk

In [ ]:
with open(resouces["rustext"], "rb") as f:
    data = f.read()

bwtChunk.write_decoded(data, "encoded/bwt/rustext", 1024 * 128)
# new_data = bwtChunk.read_decoded("encoded/bwt/rustext")

data == new_data